In [177]:
import os
import glob
import seaborn as sns
import re
import pandas as pd
import plotly.express as px
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

# hbn-specific libraries - make sure you have installed (pipenv install) and activated (pipenv shell) 
# the virtual environment for this project, and make sure you have created an ipykernel for this environment (ipython kernel install --name "hbn" --user)
from hbn.constants import Defaults
from hbn.scripts import preprocess_phenotype, make_phenotype_specs
from hbn.data import make_dataset
from hbn.features import build_features
from hbn.features.feature_selection import phenotype_features

%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [194]:
# RUN THIS CELL
# FUNCTIONS

def get_data(
    participants, 
    feature_spec, 
    cols_to_keep=['DX_01', 'DX_01_Cat', 'Age', 'Sex', 'Identifiers']
    ):
    # get data
    df = phenotype_features(target_spec=None,
                            feature_spec=os.path.join(Defaults.FEATURE_DIR, feature_spec),
                            participants=participants,
                            preprocess=False,
                            drop_identifiers=False
                            )
    
    # get summary of clinical diagnosis + other demographics
    dx = make_dataset.make_summary(save=False)
    dx = make_dataset._add_race_ethnicity(dataframe=dx)

    # get data from intake interview and merge with clinical summary
    df = df.merge(dx[cols_to_keep], on='Identifiers')
    
    return df

def get_full_dataframe(
    feature_spec,
    participants,
    cols_to_keep='Identifiers|DX_01|Age|Sex|DX_01_Cat',
    filter_scores=True,
    ):

    # get data
    list_of_cols = list(cols_to_keep.split('|'))
    df = get_data(participants, 
                 feature_spec=feature_spec,
                 cols_to_keep=list_of_cols
                 )
    
    # get assessment, domain, measure names
    _, assessment, domain, measure, _ = Path(feature_spec).name.split('-')
    
    df_all = pd.DataFrame()
    if len(df.columns) > len(list_of_cols):

        # get abbrev from dataframe based on default columns
        abbrev = df.filter(regex='Site|EID|START_DATE|Data_entry|Year').columns[0].split(',')[0]

        # load data dictionary
        dict_df = pd.read_excel(os.path.join(Defaults.PHENO_DIR, 'Release9_DataDic', abbrev + '.xlsx'), header=1) 

        # remove prefix from variable values - always second column in data dic
        dict_df.rename(columns={dict_df.columns[0]: "Question", dict_df.columns[1]: "Variable"}, inplace=True)

        # filter dataframe on certain columns and regex patterns
        df_filter = df.filter(regex=f'{abbrev}|{cols_to_keep}')

        # loop over diagnosis groups and melt `T_scores` column into one
        # concat each group to one dataframe
        for name, group in df_filter.groupby('DX_01'):
            if filter_scores:
                scores_to_filter = '_T|_Stnd|_Sum|_Score|_Scale|_Standard|_IN|_HY'
                group = group.filter(regex=f'{scores_to_filter}|{cols_to_keep}')
            tmp = group.melt(id_vars=list(cols_to_keep.split('|'))).rename({'variable':'Name', 'value': 'Scores'}, axis=1)
            tmp['Name'] = tmp['Name'].str.replace(f'{abbrev},','')
            tmp = tmp.merge(dict_df[['Question', 'Variable']], left_on=['Name'], right_on=['Variable'])
            tmp['Assessment'], tmp['domain'], tmp['measure'] = assessment, domain, measure
            df_all = pd.concat([tmp, df_all])

        # do some clean up on existing columns
        df_all['Age_rounded'] = df_all['Age'].round()
        df_all['Question'] = df_all['Question'].str.replace("T Score", "T-Score")
        
    
    return df_all.reset_index(drop=True)


In [195]:
## RUN THIS CELL ##

# Preprocess data
#preprocess_phenotype.run()

# get specs
#make_phenotype_specs.run()


# INPUTS
participants = make_dataset.get_participants(
                            split='all', 
                            disorders=['ADHD-Combined Type', 
                                        'ADHD-Inattentive Type', 
                                        'ADHD-Hyperactive_Impulsive_Type', 
                                        'Other_Specified_Attention-Deficit_Hyperactivity_Disorder',
                                        'No_Diagnosis_Given']
                                        )
# set assessment
assessment = 'Child Measures'

domains = build_features.get_domains(assessment=assessment)
domains[assessment].remove('all')
domains = [os.path.join('_'.join(re.split(r'_|,|/| ', d))) for d in domains[assessment]]

# loop over domains
feature_specs = glob.glob(os.path.join(Defaults.FEATURE_DIR, f'*{domain}*'))

# loop over feature specs 
for feature_spec in feature_specs:
    df = get_full_dataframe(
                        feature_spec=feature_spec,
                        participants=participants,
                        cols_to_keep='Identifiers|DX_01|Age|Sex|DX_01_Cat',
                        filter_scores=True,
                        )
    print(feature_spec)


ValueError: too many values to unpack (expected 5)

In [193]:
domain

'Physical_Fitness_and_Status'